In [1]:
import mlflow 
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-51-20-104-217.eu-north-1.compute.amazonaws.com:5000/")

In [2]:
# Set or create an experiment
mlflow.set_experiment("TfIdf trigram max_features")

2025/11/22 17:29:44 INFO mlflow.tracking.fluent: Experiment with name 'TfIdf trigram max_features' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://my-s3-bucket-of-store-artifact-youtube-data12/mlflow-artifacts/5', creation_time=1763812783560, experiment_id='5', last_update_time=1763812783560, lifecycle_stage='active', name='TfIdf trigram max_features', tags={}>

In [3]:
import pandas as pd
df=pd.read_csv('sentiment_clean.csv')

In [5]:
df['sentiment_numeric']=df.pop('sentiment_numeric')

In [6]:
df.head()

,text_clean,word_count,num_stop_words,num_chars,num_punctuation_chars,category_gaming,category_movies,category_music,category_technology,hour,...,anger,anticipation,trust,surprise,positive,negative,sadness,disgust,joy,sentiment_numeric
0,all products can be found on since i review 5...,24,9,116,1,0.0,0.0,0.0,1.0,19,...,0.0,0.0,0.333333,0.0,0.333333,0.0,0.0,0.0,0.333333,1
1,bro how to talk to woman in 6 steps is so rela...,12,5,53,0,0.0,0.0,0.0,1.0,23,...,0.0,0.0,0.000000,0.0,1.000000,0.0,0.0,0.0,0.000000,0
2,i was gonna say does it give you the drinks fo...,12,7,54,1,0.0,0.0,0.0,1.0,16,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0
3,anyone gonna talk abt what was o. his pc,9,3,40,1,0.0,0.0,0.0,1.0,22,...,0.0,0.0,0.000000,0.0,1.000000,0.0,0.0,0.0,0.000000,0
4,how is everyone not talking about his search?!...,15,6,85,4,0.0,0.0,0.0,1.0,12,...,0.0,0.0,0.000000,0.0,0.500000,0.0,0.0,0.0,0.500000,-1


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import scipy.sparse as sp
import numpy as np

# -----------------------------
# 1️⃣ Numeric features
# -----------------------------

X_numeric = df.iloc[:,1:-1]
y = df['sentiment_numeric']

# Scale numeric features
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_numeric)

# -----------------------------
# 2️⃣ Train-test split (reuse same split)
# -----------------------------
X_train_num, X_test_num, y_train, y_test, train_idx, test_idx = train_test_split(
    X_numeric_scaled, y, df.index, test_size=0.2, random_state=42, stratify=y
)

# -----------------------------
# 3️⃣ Text features
# -----------------------------
df_train_text = df.loc[train_idx, 'text_clean']
df_test_text = df.loc[test_idx, 'text_clean']

# -----------------------------
# 4️⃣ Experiment 3: Tune max_features
# -----------------------------
best_accuracy = 0
best_run_info = {}

def run_experiment(vectorizer_type, ngram_range, vectorizer_max_features, n_estimators=200, max_depth=15):
    global best_accuracy, best_run_info

    # Vectorization
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)

    # Fit and transform text
    X_train_vec = vectorizer.fit_transform(df_train_text)
    X_test_vec = vectorizer.transform(df_test_text)

    # Combine with numeric features
    X_train_sparse = sp.hstack([X_train_vec, sp.csr_matrix(X_train_num)])
    X_test_sparse = sp.hstack([X_test_vec, sp.csr_matrix(X_test_num)])

    # -----------------------------
    # MLflow logging
    # -----------------------------
    with mlflow.start_run() as run:
        run_name = f"{vectorizer_type}_{ngram_range}_{vectorizer_max_features}feat"
        mlflow.set_tag("mlflow.runName", run_name)
        mlflow.set_tag("experiment_type", "max_features_tuning")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Log parameters
        mlflow.log_params({
            "vectorizer_type": vectorizer_type,
            "ngram_range": ngram_range,
            "vectorizer_max_features": vectorizer_max_features,
            "numeric_features_count": X_train_num.shape[1],
            "n_estimators": n_estimators,
            "max_depth": max_depth
        })

        # Train RandomForest
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=2,
            max_features='sqrt',
            random_state=42,
            class_weight='balanced',
            n_jobs=-1
        )
        model.fit(X_train_sparse, y_train)

        # Predictions & metrics
        y_pred = model.predict(X_test_sparse)
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8,6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: {vectorizer_type}, {vectorizer_max_features} features")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log model
        mlflow.sklearn.log_model(model, f"random_forest_model_{vectorizer_type}_{vectorizer_max_features}feat")

        # Update best run
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_run_info = {
                "vectorizer": vectorizer_type,
                "ngram_range": ngram_range,
                "max_features": vectorizer_max_features,
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "accuracy": accuracy,
                "run_id": run.info.run_id
            }


# -----------------------------
# 5️⃣ Run experiments for different max_features
# -----------------------------
max_features_list = [2000, 5000, 8000, 10000, None]  # None = all features
for max_feat in max_features_list:
    run_experiment(
        vectorizer_type="TF-IDF",
        ngram_range=(1,1),
        vectorizer_max_features=max_feat,
        n_estimators=100,
        max_depth=15
    )

print(" Best Run Info:")
print(best_run_info)


2025/11/22 17:43:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/22 17:44:04 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TF-IDF_(1, 1)_2000feat at: http://ec2-51-20-104-217.eu-north-1.compute.amazonaws.com:5000/#/experiments/5/runs/56e20d2e0c9044f49623cd793a1f4b39
🧪 View experiment at: http://ec2-51-20-104-217.eu-north-1.compute.amazonaws.com:5000/#/experiments/5


2025/11/22 17:53:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/22 17:54:09 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TF-IDF_(1, 1)_5000feat at: http://ec2-51-20-104-217.eu-north-1.compute.amazonaws.com:5000/#/experiments/5/runs/11f9c46a74a9431fb4971d9335203f07
🧪 View experiment at: http://ec2-51-20-104-217.eu-north-1.compute.amazonaws.com:5000/#/experiments/5


2025/11/22 17:54:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/22 17:55:23 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TF-IDF_(1, 1)_8000feat at: http://ec2-51-20-104-217.eu-north-1.compute.amazonaws.com:5000/#/experiments/5/runs/461e3cf6845740a09e2ec228bc08c1be
🧪 View experiment at: http://ec2-51-20-104-217.eu-north-1.compute.amazonaws.com:5000/#/experiments/5


2025/11/22 17:55:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/22 17:56:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TF-IDF_(1, 1)_10000feat at: http://ec2-51-20-104-217.eu-north-1.compute.amazonaws.com:5000/#/experiments/5/runs/7733540af45d409f8f5aca7c20fade58
🧪 View experiment at: http://ec2-51-20-104-217.eu-north-1.compute.amazonaws.com:5000/#/experiments/5


2025/11/22 17:58:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/22 17:58:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TF-IDF_(1, 1)_Nonefeat at: http://ec2-51-20-104-217.eu-north-1.compute.amazonaws.com:5000/#/experiments/5/runs/30265f9d02f14e88bdcf17cb087771ae
🧪 View experiment at: http://ec2-51-20-104-217.eu-north-1.compute.amazonaws.com:5000/#/experiments/5
 Best Run Info:
{'vectorizer': 'TF-IDF', 'ngram_range': (1, 1), 'max_features': 2000, 'n_estimators': 100, 'max_depth': 15, 'accuracy': 0.6678455941794664, 'run_id': '56e20d2e0c9044f49623cd793a1f4b39'}
